### Import statements

In [1]:
import os
import csv
import re
import git
import bisect
from copy import deepcopy
import requests
from lxml import etree
import logging
import pandas as pd
import spacy
from MyCapytain.resolvers.cts.local import CtsCapitainsLocalResolver
from MyCapytain.resources.prototypes.metadata import UnknownCollection

### Global values

In [2]:
data_dir = "data"
git_base_url = "http://github.com/PerseusDL"
repo_names = ["canonical-greekLit"]
nsmap = {
    "cts": "http://chs.harvard.edu/xmlns/cts",
    "tei": "http://www.tei-c.org/ns/1.0",
    "py": "http://codespeak.net/lxml/objectify/pytype",
}
spacy_model = "grc_odycy_joint_trf"

### Create local clone of Perseus repository

In [3]:
print('Checking for local text repositories...')
for repo in repo_names:
    dest_path = os.path.join(data_dir, repo)
    remote_url = f"{git_base_url}/{repo}.git"

    if os.path.exists(dest_path):
        print(f' - {dest_path} exists!')
    else:
        print(f' - retrieving {dest_path}')
        git.Repo.clone_from(remote_url, dest_path)

Checking for local text repositories...
 - data/canonical-greekLit exists!


### Read text metadata

In [4]:
corpus_file = os.path.join(data_dir, "corpus.tsv")
print(f"Reading text metadata from {corpus_file}")

corpus = []

with open(corpus_file) as f:
    reader = csv.DictReader(f, delimiter="\t")
    for rec in reader:
        corpus.append(dict(
            author = rec["author"],
            title = rec["title"],
            urn = rec["urn"],
            prefix = rec["prefix"],
        ))
print(f" - got data on {len(corpus)} texts")

Reading text metadata from data/corpus.tsv
 - got data on 115 texts


### Load XML using CTS API

#### initialize local resolver

In [5]:
repo_paths = [os.path.join(data_dir, repo) for repo in repo_names]
resolver = CtsCapitainsLocalResolver(repo_paths)

#### retrieve the texts

In [10]:
print("Retrieving texts...")
for text in corpus:
    try:
        text["xml"] = resolver.getTextualNode(text["urn"], text["prefix"]).xml
    except UnknownCollection:
        print(f" - {text['author']} {text['title']} {text['prefix']} failed")

print(len([text for text in corpus if text.get("xml") is not None]), "texts retrieved")

Retrieving texts...
115 texts retrieved


### Extract verse lines

#### build an array of lines

In [21]:
print("Building line arrays...")
for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")

    # bail if no XML
    if text.get("xml") is None:
        print("failed: no XML")
        continue

    # initialize line array
    text['line_array'] = []

    # work from copy of xml
    xml = deepcopy(text["xml"])
        
    # remove notes
    for note in xml.findall(".//tei:note", namespaces=nsmap):
        note.clear(keep_tail=True)
        
    # remove deleted lines
    for del_ in xml.findall(".//tei:del", namespaces=nsmap):
        del_.clear(keep_tail=True)
        
    # iterate over verse lines
    for l in xml.findall(".//tei:l", namespaces=nsmap):
        line_num = l.get("n")
        line_text = "".join(s for s in l.itertext())
        line_text = re.sub(r"\s+", " ", line_text).strip()
        text['line_array'].append(dict(
            n = line_num,
            text = line_text,
        ))

    print(len(text['line_array']), "lines")

Building line arrays...
[1/115] Homer Iliad 1	611 lines
[2/115] Homer Iliad 2	877 lines
[3/115] Homer Iliad 3	461 lines
[4/115] Homer Iliad 4	544 lines
[5/115] Homer Iliad 5	909 lines
[6/115] Homer Iliad 6	529 lines
[7/115] Homer Iliad 7	482 lines
[8/115] Homer Iliad 8	565 lines
[9/115] Homer Iliad 9	709 lines
[10/115] Homer Iliad 10	579 lines
[11/115] Homer Iliad 11	847 lines
[12/115] Homer Iliad 12	471 lines
[13/115] Homer Iliad 13	837 lines
[14/115] Homer Iliad 14	521 lines
[15/115] Homer Iliad 15	746 lines
[16/115] Homer Iliad 16	867 lines
[17/115] Homer Iliad 17	761 lines
[18/115] Homer Iliad 18	617 lines
[19/115] Homer Iliad 19	424 lines
[20/115] Homer Iliad 20	503 lines
[21/115] Homer Iliad 21	611 lines
[22/115] Homer Iliad 22	515 lines
[23/115] Homer Iliad 23	897 lines
[24/115] Homer Iliad 24	804 lines
[25/115] Homer Odyssey 1	444 lines
[26/115] Homer Odyssey 2	434 lines
[27/115] Homer Odyssey 3	497 lines
[28/115] Homer Odyssey 4	847 lines
[29/115] Homer Odyssey 5	493 lines
[30

#### index lines by starting character position

In [19]:
print("Building line indices...")
success = 0

for text in corpus:
    # start index at zero
    text["line_index"] = []
    cumsum = 0

    # iterate over line array, add length (plus 1 for spaces between lines)
    for line in text["line_array"]:
        text["line_index"].append(cumsum)
        cumsum += len(line["text"]) + 1

    # make sure the count works out
    if cumsum != len(" ".join(line["text"] for line in text["line_array"])) + 1:
        print(" - character count doesn't match:", text["author"], text["title"], text["prefix"])
    else:
        success += 1

print(success, " indices built successfully")

Building line indices...
115  indices built successfully


### Parse text with spaCy
#### load spacy model

In [22]:
nlp = spacy.load(spacy_model)

#### parse each text as one long string

In [24]:
print(f"Parsing with {spacy_model}...")

for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")

    one_long_string = " ".join(line["text"] for line in text["line_array"])
    text["spacy_doc"] = nlp(one_long_string)

    print(len(text["spacy_doc"]), "tokens")
    

Parsing with grc_odycy_joint_trf...
[1/115] Homer Iliad 1	5140 tokens
[2/115] Homer Iliad 2	6822 tokens
[3/115] Homer Iliad 3	3686 tokens
[4/115] Homer Iliad 4	4341 tokens
[5/115] Homer Iliad 5	7306 tokens
[6/115] Homer Iliad 6	4247 tokens
[7/115] Homer Iliad 7	3867 tokens
[8/115] Homer Iliad 8	4560 tokens
[9/115] Homer Iliad 9	5778 tokens
[10/115] Homer Iliad 10	4751 tokens
[11/115] Homer Iliad 11	6860 tokens
[12/115] Homer Iliad 12	3741 tokens
[13/115] Homer Iliad 13	6684 tokens
[14/115] Homer Iliad 14	4241 tokens
[15/115] Homer Iliad 15	6063 tokens
[16/115] Homer Iliad 16	6970 tokens
[17/115] Homer Iliad 17	6134 tokens
[18/115] Homer Iliad 18	5030 tokens
[19/115] Homer Iliad 19	3460 tokens
[20/115] Homer Iliad 20	4080 tokens
[21/115] Homer Iliad 21	5074 tokens
[22/115] Homer Iliad 22	4326 tokens
[23/115] Homer Iliad 23	7318 tokens
[24/115] Homer Iliad 24	6838 tokens
[25/115] Homer Odyssey 1	3733 tokens
[26/115] Homer Odyssey 2	3727 tokens
[27/115] Homer Odyssey 3	4213 tokens
[28/115